# 04 — Fuzzy join: The Numbers → spine (Phase 5)

**Inputs:** `data/interim/spine.csv` (1,189 films, keyed by imdb_id) +
`data/interim/the_numbers_union.csv` (3,812 films, no imdb_id — title+year only).

**Output:** `data/interim/analysis_table.csv` — spine + nominal budget/grosses
for matched films, with a `match_method` audit column.

**Matching ladder** (each tier only sees films the previous tiers missed):
1. `exact` — normalized title + same year
2. `year_pm1` — normalized title + year ±1 (release-date vs film-year slop)
3. `fuzzy` — rapidfuzz token_sort_ratio ≥ 92 within year ±1, auto-accepted
4. `review band` — scores 80–92 are NOT accepted; they're printed for hand-check

**Validation:** match rate; 30-pair eyeball sample; selection check on
unmatched; and a cross-source gross sanity correlation.

In [1]:
import re
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz, process

INTERIM = Path.cwd().parent / "data" / "interim"

spine = pd.read_csv(INTERIM / "spine.csv")
numbers = pd.read_csv(INTERIM / "the_numbers_union.csv")
numbers = numbers.dropna(subset=["year"]).copy()
numbers["year"] = numbers["year"].astype(int)

def norm_title(s):
    """lowercase; strip punctuation but KEEP word boundaries (fuzzy scorers need them)"""
    s = s.lower().replace("&", " and ")  # "&" vs "and" caused systematic near-misses
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

spine["t"] = spine["title"].map(norm_title)
numbers["t"] = numbers["movie"].map(norm_title)

print(f"spine {len(spine)} | numbers {len(numbers)}")

spine 1189 | numbers 3810


## Tiers 1–2: exact and ±1-year key matches

In [2]:
num_cols = ["movie", "year", "production_budget", "domestic_gross",
            "worldwide_gross", "distributor", "mpaa_rating", "genre", "source", "t"]

def key_match(sp, nums, year_offset):
    n = nums[num_cols].copy()
    n["year_key"] = n["year"] + year_offset
    m = sp.merge(n, left_on=["t", "year"], right_on=["t", "year_key"],
                 suffixes=("", "_num"))
    return m.drop(columns="year_key")

# tier 1: same year
t1 = key_match(spine, numbers, 0)
t1["match_method"] = "exact"
matched_ids = set(t1["imdb_id"])

# tier 2: numbers year one off in either direction (first hit wins)
rest = spine[~spine["imdb_id"].isin(matched_ids)]
t2 = pd.concat([key_match(rest, numbers, +1), key_match(rest, numbers, -1)])\
    .drop_duplicates("imdb_id", keep="first")
t2["match_method"] = "year_pm1"
matched_ids |= set(t2["imdb_id"])

print(f"tier 1 exact:    {len(t1):,}")
print(f"tier 2 year±1:   {len(t2):,}")
print(f"still unmatched: {len(spine) - len(matched_ids):,}")

tier 1 exact:    741
tier 2 year±1:   42
still unmatched: 406


## Tier 3: rapidfuzz on the residue

For each unmatched spine film, score it against Numbers titles within ±1 year.
`token_sort_ratio` is order-insensitive ("Two Towers, The" ≈ "The Two Towers").
≥ 92 auto-accepts; 80–92 goes to the review table below — accepted only by
explicit human decision, never silently.

In [3]:
rest = spine[~spine["imdb_id"].isin(matched_ids)]

auto, review = [], []
for _, film in rest.iterrows():
    cand = numbers[numbers["year"].sub(film["year"]).abs() <= 1]
    if cand.empty:
        continue
    hit = process.extractOne(film["t"], cand["t"], scorer=fuzz.token_sort_ratio,
                             score_cutoff=80)
    if hit is None:
        continue
    text, score, idx = hit  # idx is the numbers-frame index LABEL (choices was a Series)
    row = {"imdb_id": film["imdb_id"], "spine_title": film["title"],
           "spine_year": film["year"], "num_idx": idx,
           "num_title": numbers.loc[idx, "movie"], "num_year": int(numbers.loc[idx, "year"]),
           "score": round(score, 1)}
    (auto if score >= 92 else review).append(row)

print(f"fuzzy auto-accept (>=92): {len(auto)}")
print(f"review band (80-92):      {len(review)}")
pd.DataFrame(auto).head(10)

fuzzy auto-accept (>=92): 9
review band (80-92):      12


,imdb_id,spine_title,spine_year,num_idx,num_title,num_year,score
0,tt0212720,A.I. Artificial Intelligence,2001,539,Artificial Intelligence: AI,2001,94.3
1,tt0251736,House of 1000 Corpses,2003,3076,"House of 1,000 Corpses",2003,93.0
2,tt0116583,Hunchback of Notre Dame,1996,455,The Hunchback of Notre Dame,1996,92.0
3,tt0120755,Mission: Impossible II,2000,348,Mission: Impossible 2,2000,92.7
4,tt0473075,Prince of Persia: The Sands of Time,2010,64,Prince of Persia: Sands of Time,2010,93.8
5,tt0120915,Star Wars: Episode I - The Phantom Menace,1999,364,Star Wars Ep. I: The Phantom Menace,1999,93.0
6,tt0121765,Star Wars: Episode II - Attack of the Clones,2002,358,Star Wars Ep. II: Attack of the Clones,2002,93.5
7,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005,357,Star Wars Ep. III: Revenge of the Sith,2005,93.5
8,tt2488496,Star Wars: Episode VII - The Force Awakens,2015,5,Star Wars Ep. VII: The Force Awakens,2015,93.2


In [4]:
review_df = pd.DataFrame(review)
print("REVIEW BAND — hand-check these; accept by adding imdb_id to ACCEPT below")
review_df

REVIEW BAND — hand-check these; accept by adding imdb_id to ACCEPT below


,imdb_id,spine_title,spine_year,num_idx,num_title,num_year,score
0,tt0103644,Alien³,1992,1111,Alien 3,1992,83.3
1,tt1024648,Argo,2012,3338,Cargo,2011,88.9
2,tt0276816,Below,2002,1747,Blow,2001,88.9
3,tt0317248,City of God,2002,2338,City of Ghosts,2003,80.0
4,tt0130018,I Still Know What You Did Last Summer,1998,2367,I Know What You Did Last Summer,1997,91.2
5,tt0378194,Kill Bill: Vol. 2,2004,1091,Kill Bill: Volume 2,2004,90.9
6,tt0325980,Pirates of the Caribbean: The Curse of the Bla...,2003,263,Pirates of the Caribbean: The Curse of the Bla...,2003,83.6
7,tt0363771,"The Chronicles of Narnia: The Lion, the Witch ...",2005,115,"The Chronicles of Narnia: The Lion, the Witch ...",2005,81.0
8,tt0118971,The Devil's Advocate,1997,1061,Devil's Advocate,1997,88.9
9,tt0119174,The Game,1997,1958,He Got Game,1998,84.2


In [5]:
# Hand-check decisions: after eyeballing the review table, list imdb_ids of
# pairs that ARE the same film. Empty until a human has looked.
# Hand-checked 2026-07-05 (see review table above): same film, different
# title stylization. Rejected as DIFFERENT films: Argo/Cargo, Below/Blow,
# City of God/City of Ghosts, The Game/He Got Game, The Past/The East,
# I Still Know.../I Know What You Did Last Summer (sequel vs original).
ACCEPT = [
    "tt0103644",  # Alien³ = Alien 3 (1992)
    "tt0378194",  # Kill Bill: Vol. 2 = Kill Bill: Volume 2 (2004)
    "tt0325980",  # Pirates of the Caribbean: COTBP (2003)
    "tt0363771",  # Chronicles of Narnia: LWW (2005)
    "tt0118971",  # The Devil's Advocate = Devil's Advocate (1997)
    "tt0245674",  # Thir13en Ghosts = Thirteen Ghosts (2001)
    # "&"-variant pairs now match in the exact tier via norm_title fix
]

accepted = [r for r in auto] + [r for r in review if r["imdb_id"] in ACCEPT]
t3 = pd.DataFrame(accepted)
if len(t3):
    t3 = t3.merge(spine, on="imdb_id", suffixes=("_m", ""))\
           .merge(numbers[num_cols].reset_index(names="num_idx"), on="num_idx",
                  suffixes=("", "_num"))
    t3["match_method"] = "fuzzy"
    t3 = t3[t1.columns]
print(f"tier 3 fuzzy accepted: {len(t3)}")

tier 3 fuzzy accepted: 15


## Assemble + audit the matched table

In [6]:
matched = pd.concat([t1, t2, t3], ignore_index=True)

# audit: no spine film twice, and no Numbers row claimed by two spine films
assert matched["imdb_id"].is_unique, "a spine film matched twice"
dupe_claims = matched.duplicated(["t", "year_num" if "year_num" in matched else "movie"], keep=False)
print(f"matched: {len(matched):,} of {len(spine):,} spine films "
      f"({len(matched)/len(spine):.1%})")
print(matched["match_method"].value_counts())
print(f"\nNumbers rows claimed by >1 spine film: {matched.duplicated(['movie','year_num'] if 'year_num' in matched.columns else ['movie']).sum()}")

matched: 798 of 1,189 spine films (67.1%)
match_method
exact       741
year_pm1     42
fuzzy        15
Name: count, dtype: int64

Numbers rows claimed by >1 spine film: 0


## Validation 1 — eyeball 30 matched pairs

Random sample across tiers. Look for: same film? plausible budget for the
title? (Errors cluster in remakes, sequels, and generic titles.)

In [7]:
cols = ["title", "movie", "year", "match_method", "production_budget", "domestic_gross"]
matched.sample(min(30, len(matched)), random_state=530)[cols]\
    .sort_values("match_method")

,title,movie,year,match_method,production_budget,domestic_gross
38,Antz,Antz,1998,exact,60000000,90757863
578,The Good Shepherd,The Good Shepherd,2006,exact,80000000,59908565
254,Hot Fuzz,Hot Fuzz,2007,exact,16000000,23618786
183,Final Destination 3,Final Destination 3,2006,exact,25000000,54098051
448,Rush,Rush,2013,exact,38000000,26947624
470,Serenity,Serenity,2005,exact,39000000,25514517
478,Shrek 2,Shrek 2,2004,exact,70000000,441226247
411,Pirates of the Caribbean: Dead Man's Chest,Pirates of the Caribbean: Dead Manâs Chest,2006,exact,225000000,423315812
322,Lord of War,Lord of War,2005,exact,42000000,24149632
292,Jumper,Jumper,2008,exact,82500000,80172128


## Validation 2 — cross-source gross correlation

Numbers `domestic_gross` (nominal) and Pudding `gross_infl_adj_musd`
(inflation-adjusted) measure the same quantity on different bases. On matched
pairs their log-log correlation should be very high; a low value would mean
the join is pairing different films.

In [8]:
import numpy as np

ok = (matched["domestic_gross"] > 0) & (matched["gross_infl_adj_musd"] > 0)
r = np.corrcoef(np.log10(matched.loc[ok, "domestic_gross"]),
                np.log10(matched.loc[ok, "gross_infl_adj_musd"]))[0, 1]
print(f"log-log correlation on {ok.sum()} matched films with positive gross: r = {r:.3f}")
print(f"films with domestic_gross == 0 in matched set: {(matched['domestic_gross'] == 0).sum()} (flagged, not dropped here)")

log-log correlation on 787 matched films with positive gross: r = 0.981
films with domestic_gross == 0 in matched set: 0 (flagged, not dropped here)


## Validation 3 — selection check on the unmatched

In [9]:
unmatched = spine[~spine["imdb_id"].isin(matched["imdb_id"])]
cmp = pd.DataFrame({
    "matched": matched[["year", "gross_infl_adj_musd", "pct_female_dialogue", "bechdel_rating"]].mean(),
    "unmatched": unmatched[["year", "gross_infl_adj_musd", "pct_female_dialogue", "bechdel_rating"]].mean(),
}).round(3)
print(f"unmatched spine films: {len(unmatched)}")
print(cmp)
print("\nunmatched sample:")
unmatched[["imdb_id", "title", "year", "gross_infl_adj_musd"]].sample(min(10, len(unmatched)), random_state=530)

unmatched spine films: 391
                      matched  unmatched
year                 2005.147   2004.107
gross_infl_adj_musd   124.942     63.852
pct_female_dialogue     0.283      0.336
bechdel_rating          2.048      2.169

unmatched sample:


,imdb_id,title,year,gross_infl_adj_musd
289,tt0106834,Even Cowgirls Get the Blues,1993,3.0
898,tt0104040,The Cutting Edge,1992,52.0
611,tt0104990,Newsies,1992,5.0
425,tt0130018,I Still Know What You Did Last Summer,1998,74.0
739,tt0108065,Searching for Bobby Fischer,1993,15.0
521,tt2692904,Locke,2013,1.0
943,tt1601913,The Grey,2011,56.0
836,tt1397280,Taken 2,2012,153.0
899,tt0838221,The Darjeeling Limited,2007,15.0
1070,tt1142988,The Ugly Truth,2009,103.0


## Write the analysis table

In [10]:
drop_helpers = [c for c in ["t", "t_num", "movie", "year_num"] if c in matched.columns]
analysis = matched.drop(columns=drop_helpers)
out_path = INTERIM / "analysis_table.csv"
analysis.to_csv(out_path, index=False)
print(f"wrote {len(analysis):,} rows x {analysis.shape[1]} cols -> {out_path}")
list(analysis.columns)

wrote 798 rows x 23 cols -> /Users/trevorjmckee/Documents/Work/School/03_Summer_2026/DSC530/final_project/data/interim/analysis_table.csv


['script_id',
 'imdb_id',
 'title',
 'year',
 'gross_infl_adj_musd',
 'total_words',
 'f_words',
 'n_characters',
 'n_female',
 'pct_female_dialogue',
 'pct_female_characters',
 'female_lead',
 'bechdel_rating',
 'bechdel_year',
 'bechdel_pass',
 'production_budget',
 'domestic_gross',
 'worldwide_gross',
 'distributor',
 'mpaa_rating',
 'genre',
 'source',
 'match_method']

## Findings log

- **Match rate: 798 of 1,189 (67.1%)** — exact 741 / year±1 42 / fuzzy 15
  (9 auto ≥92, 6 hand-accepted from the review band; decisions recorded in `ACCEPT`)
- **Systematic fix found during review:** "&" vs "and" caused near-misses;
  normalized in `norm_title` rather than hand-patched
- **Cross-source gross correlation: r = 0.981** (log-log, n=787) — matched pairs
  are overwhelmingly the same films
- **Selection — the important one:** unmatched spine films are smaller
  (**$64M vs $125M** mean infl-adj gross) and — notably — have HIGHER female
  dialogue share (**33.6% vs 28.3%**) and higher mean Bechdel rating
  (2.17 vs 2.05). The budget requirement preferentially drops smaller,
  more female-forward films. First-class limitation: the final analysis sample
  is the *intersection* of three coverage filters, each skewing toward big
  commercial films.
- `domestic_gross == 0`: 0 rows in the matched table — no handling needed
- Nominal-vs-adjusted rule holds: model uses Numbers (nominal) OR Pudding
  (adjusted) gross, never both as one variable